# 8.1 期货基础：升贴水、展期与 Roll Yield

## 学习目标
- 理解期货合约的基本结构
- 掌握持有成本模型（Cost of Carry）
- 计算 Roll Yield（展期收益）
- 区分 Contango 和 Backwardation


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
np.random.seed(42)
print('Libraries loaded')


## 1. 期货定价：持有成本模型

$$F_{t,T} = S_t \cdot e^{(r + c - y)(T-t)}$$

- $S_t$：现货价格
- $r$：无风险利率
- $c$：储存成本（正数，如仓储费）
- $y$：便利收益率（正数，如持有实物的便利性）
- $(T-t)$：剩余到期时间

**Contango（期货溢价）**：$F > S$，期货价格高于现货（常见于金融资产、石油）
**Backwardation（期货贴水）**：$F < S$，期货价格低于现货（常见于农产品丰收季）


In [ ]:
S0 = 100      # 现货价格
r  = 0.04     # 无风险利率
c  = 0.02     # 储存成本（年化）
y  = 0.01     # 便利收益率（年化）

T_values = np.linspace(0.1, 2.0, 100)  # 0.1 到 2 年

# Contango 场景（c > y → F > S）
F_contango = S0 * np.exp((r + c - y) * T_values)
# Backwardation 场景（y > r + c → F < S）
F_backwardation = S0 * np.exp((r + 0.005 - 0.06) * T_values)

plt.figure(figsize=(10, 5))
plt.plot(T_values, F_contango, 'r-', lw=2, label='Contango（期货溢价）')
plt.plot(T_values, F_backwardation, 'b-', lw=2, label='Backwardation（期货贴水）')
plt.axhline(S0, color='gray', linestyle='--', lw=1.5, label=f'现货价格 S0={S0}')
plt.xlabel('到期时间（年）'); plt.ylabel('期货价格')
plt.title('期货期限结构：Contango vs Backwardation')
plt.legend(); plt.grid(alpha=0.3); plt.show()


## 2. Roll Yield（展期收益）

**期货投资者最重要的误解**：期货的实际持有收益 ≠ 现货价格变化。

期货投资者必须定期将到期合约展期（Roll）到下一个合约，这会产生额外的盈亏：

$$\text{Roll Yield} = \frac{F_{near} - F_{far}}{F_{near}}$$

- Backwardation 中：$F_{near} > F_{far}$ → Roll Yield > 0（每次展期获利）
- Contango 中：$F_{near} < F_{far}$ → Roll Yield < 0（每次展期亏损）


In [ ]:
# 模拟 12 个月的展期成本
months = 12
# 假设市场处于 Contango：近月 < 远月
near_futures = 100 + np.random.normal(0, 2, months)  # 近月合约价格
far_futures  = near_futures + np.random.uniform(1, 4, months)  # 远月溢价 1-4 元

roll_yield = (near_futures - far_futures) / near_futures  # 负数=成本
spot_return = np.random.normal(0.005, 0.02, months)       # 现货收益率
total_futures_return = spot_return + roll_yield            # 期货总收益 = 现货 + 展期

plt.figure(figsize=(10, 5))
width = 0.35
x = np.arange(months)
plt.bar(x - width/2, spot_return*100, width, label='现货价格收益率', color='steelblue', alpha=0.8)
plt.bar(x + width/2, roll_yield*100, width, label='展期成本（Roll Yield）', color='red', alpha=0.8)
plt.plot(x, total_futures_return*100, 'ko-', ms=5, label='期货总收益率', lw=1.5)
plt.axhline(0, color='black', lw=0.8)
plt.title('Contango 期货：展期成本大幅侵蚀现货收益')
plt.xlabel('月份'); plt.ylabel('月收益率 (%)')
plt.legend(); plt.grid(alpha=0.3); plt.show()
print(f'年化现货收益: {spot_return.sum()*100:.2f}%')
print(f'年化展期成本: {roll_yield.sum()*100:.2f}%')
print(f'年化期货总收益: {total_futures_return.sum()*100:.2f}%')


## 🎯 练习

1. 对原油期货（WTI）历史数据，计算月度 Roll Yield，观察 2015-2016 年油价暴跌期间 Roll Yield 的变化。
2. 对比黄金期货（通常 Contango）和铜期货（有时 Backwardation）的历史 Roll Yield 差异。
3. 设计一个「Carry 策略」：买入 Backwardation 品种，卖空 Contango 品种，计算模拟收益。

---
**下一节** → `02_commodity_momentum.ipynb`